In [0]:
from datetime import datetime
from pyspark.sql.functions import max as spark_max

# Optimize shuffle for small clusters
spark.conf.set("spark.sql.shuffle.partitions", 1)

# 1. Widgets & Setup
dbutils.widgets.text("customer_id", "cust_001")
dbutils.widgets.text("object_name", "events")
dbutils.widgets.text("source_system", "bigquery")
dbutils.widgets.text("bucket_path", "s3://hgi-databricks-data-lakehouse-dev/")
# CHANGED: Using native table string instead of catalog table
dbutils.widgets.text("bq_native_table", "v4c-bigquery.v4c_bigquery_dataset.event_raw")

customer_id = dbutils.widgets.get("customer_id")
object_name = dbutils.widgets.get("object_name")
source_system = dbutils.widgets.get("source_system")
bucket_path = dbutils.widgets.get("bucket_path")
bq_native_table = dbutils.widgets.get("bq_native_table")

# Extract the GCP Project ID dynamically from the table string
gcp_project_id = bq_native_table.split(".")[0]

# Timestamp path
now = datetime.now()
yyyy, mm, dd, hh = now.strftime("%Y"), now.strftime("%m"), now.strftime("%d"), now.strftime("%H")
incremental_path = f"{bucket_path}landing-zone/{source_system}/{customer_id}/{object_name}/incremental/{yyyy}/{mm}/{dd}/{hh}"

print(f"🚀 Starting FAST NATIVE Incremental Load to: {incremental_path}")

# 2. Securely Retrieve GCP Credentials
try:
    gcp_secret = dbutils.secrets.get(scope="gcp_auth", key="bq_key")
except Exception as e:
    raise Exception("❌ Failed to retrieve GCP credentials. Error: " + str(e))

# 3. Get the last watermark
try:
    watermark_df = spark.sql(f"""
    SELECT last_processed_timestamp
    FROM ingestion_metadata.watermarks
    WHERE source_system='{source_system}'
    AND object_name='{object_name}'
    """)
    rows = watermark_df.collect()
    if len(rows) == 0:
        raise Exception("Watermark missing.")
    
    # This is a native Python datetime object from your Delta table
    watermark_dt = rows[0][0] 
    
    # THE FIX: Format the Python datetime into BigQuery's exact string format
    watermark_str = watermark_dt.strftime("%Y-%m-%dT%H:%M:%S.%f")[:-3] + "Z"
    
except Exception as e:
    raise Exception(f"❌ Watermark not initialized. Please run historical load first. Error: {str(e)}")

print(f"📍 Last Watermark: {watermark_str}")

# 4. Query BigQuery FIRST to get the NEW max timestamp (Native Connector)
# THE FIX: Direct string comparison in the filter
new_ts_df = spark.read \
    .format("bigquery") \
    .option("credentials", gcp_secret) \
    .option("parentProject", gcp_project_id) \
    .option("project", gcp_project_id) \
    .option("filter", f"event_timestamp > '{watermark_str}'") \
    .load(bq_native_table) \
    .select(spark_max("event_timestamp").alias("max_ts"))

# Since the BQ column is a STRING, this returns the max lexicographical string
new_ts_str = new_ts_df.first()[0]

if new_ts_str is None:
    print("✅ No new records found in BigQuery. Exiting gracefully.")
    dbutils.notebook.exit("0")

print(f"🎯 New records found up to: {new_ts_str}")

# 5. Pull the data using BOTH bounds (Native Connector)
# THE FIX: Direct string comparison in the filter
df_incremental = spark.read \
    .format("bigquery") \
    .option("credentials", gcp_secret) \
    .option("parentProject", gcp_project_id) \
    .option("project", gcp_project_id) \
    .option("filter", f"event_timestamp > '{watermark_str}' AND event_timestamp <= '{new_ts_str}'") \
    .load(bq_native_table)

try:
    # 6. Write to S3 (Adding repartition(2) for the 2-core m5d.large instance)
    df_incremental.repartition(2).write \
        .mode("append") \
        .format("parquet") \
        .option("compression", "snappy") \
        .save(incremental_path)
    print("💾 Write to S3 completed successfully.")
    
except Exception as e:
    print(f"❌ Failed during write operation: {str(e)}")
    raise e

# 7. Update Watermark with 1-Minute Lookback
print("Updating watermark table with lookback window...")

# THE FIX: We use CAST('{new_ts_str}' AS TIMESTAMP) so Databricks understands the ISO string
spark.sql(f"""
    CREATE OR REPLACE TEMP VIEW new_watermarks AS
    SELECT source_system, object_name, 
           CASE 
               WHEN source_system = '{source_system}' AND object_name = '{object_name}' THEN CAST('{new_ts_str}' AS TIMESTAMP) - INTERVAL 1 MINUTE
               ELSE last_processed_timestamp 
           END as last_processed_timestamp
    FROM ingestion_metadata.watermarks
""")

spark.sql("INSERT OVERWRITE TABLE ingestion_metadata.watermarks SELECT * FROM new_watermarks")

print(f"✅ Watermark updated successfully (minus 1 minute).")
dbutils.notebook.exit("Success")